# Banking Intent Unsloth — Fine-tune Qwen3-8B on Kaggle

**Before running:**
1. Kaggle → Settings → Secrets: add `HF_TOKEN` (write-access HuggingFace token)
2. Notebook settings → Accelerator: **GPU T4 x2** or **P100**
3. Notebook settings → Internet: **On**
4. Set `HF_REPO_ID` below to your HuggingFace repo (e.g. `nguyenvmthien/banking-intent-unsloth`)

In [ ]:
HF_REPO_ID = "nguyenvmthien/banking-intent-unsloth"  # <-- change this

## 1. Environment setup

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

# Unsloth must be installed before transformers to avoid version conflicts
print(run('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"'))
print(run('pip install -q trl peft datasets langsmith huggingface_hub pandas scikit-learn tqdm pyyaml'))
print("Dependencies installed.")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
print("HF_TOKEN loaded.")

## 2. Clone repo and prepare data

In [ ]:
REPO_URL = "https://github.com/nguyenvmthien/banking-intent-v2.git"  # <-- change this

run(f'git clone {REPO_URL} /kaggle/working/banking-intent-v2')
os.chdir('/kaggle/working/banking-intent-v2/scripts')
print("Repo cloned.")

In [ ]:
run('python preprocess_data.py')
print("Data ready.")

## 3. Configure HF repo for checkpoint push

In [ ]:
import yaml

config_path = '../configs/train.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

config['hub_model_id'] = HF_REPO_ID

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"hub_model_id set to: {HF_REPO_ID}")

## 4. Train

If this cell is re-run after a session restart, it will **automatically resume** from the latest local checkpoint (if any), or pull from Hub if disk was wiped.

In [ ]:
# If Kaggle disk was wiped, try to restore latest checkpoint from Hub
import glob
from huggingface_hub import snapshot_download

output_dir = '/kaggle/working/banking-intent-v2/outputs/checkpoint'
has_local_checkpoint = bool(glob.glob(os.path.join(output_dir, 'checkpoint-*')))

if not has_local_checkpoint:
    try:
        print(f"No local checkpoint. Trying to restore from Hub: {HF_REPO_ID}")
        snapshot_download(
            repo_id=HF_REPO_ID,
            local_dir=output_dir,
            token=os.environ["HF_TOKEN"],
        )
        print("Restored from Hub.")
    except Exception as e:
        print(f"Hub restore skipped ({e}) — starting fresh.")

In [ ]:
os.chdir('/kaggle/working/banking-intent-v2/scripts')
run_result = subprocess.run(
    [sys.executable, 'train.py'],
    env=os.environ,
)
if run_result.returncode != 0:
    raise RuntimeError("Training failed — check logs above.")
print("Training complete.")

## 5. Quick sanity check — predict one sample

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/banking-intent-v2/scripts')
from inference import IntentClassification

clf = IntentClassification('../configs/inference.yaml', mode='finetuned')
test_input = "I lost my credit card, how do I order a replacement?"
result = clf.predict(test_input)
print(result)